In [0]:
# %sql
# DROP TABLE IF EXISTS realtime_weather.silver.silver_weather_clean;
# DROP TABLE IF EXISTS realtime_weather.silver.silver_weather_quarantine;

In [0]:
# dbutils.fs.rm("/Volumes/realtime_weather/silver/checkpoint", True)

In [0]:
%sql
use catalog realtime_weather

In [0]:
%sql
create volume if not exists realtime_weather.silver.checkpoint

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

In [0]:
bronze_df = spark.readStream \
    .format("delta") \
    .table("realtime_weather.bronze.bronze_weather_data")

In [0]:
weather_schema = StructType([
    StructField("name", StringType()),
    StructField("dt", LongType()),
    
    StructField("coord", StructType([
        StructField("lat", DoubleType()),
        StructField("lon", DoubleType())
    ])),
    
    StructField("sys", StructType([
        StructField("country", StringType()),
        StructField("sunrise", LongType()),
        StructField("sunset", LongType())
    ])),
    
    StructField("main", StructType([
        StructField("temp", DoubleType()),
        StructField("humidity", IntegerType()),
        StructField("pressure", IntegerType()),
        StructField("feels_like", DoubleType())
    ])),
    
    StructField("wind", StructType([
        StructField("speed", DoubleType())
    ])),
    
    StructField("weather", ArrayType(
        StructType([
            StructField("description", StringType())
        ])
    )),
    
    StructField("rain", StructType([
        StructField("1h", DoubleType())
    ]))
])


In [0]:
parsed_df = bronze_df.withColumn(
    "parsed",
    from_json(col("raw_response"), weather_schema)
)

In [0]:
weather_df = parsed_df.select(
    col("http_status"),
    col("raw_response"),
    col("call_timestamp"),

    col("parsed.name").alias("city_name"),
    col("parsed.sys.country").alias("country"),
    col("parsed.coord.lat").alias("latitude"),
    col("parsed.coord.lon").alias("longitude"),

    col("parsed.main.temp").alias("temperature_celsius"),
    col("parsed.main.feels_like").alias("feels_like_celsius"),
    col("parsed.main.humidity").alias("humidity_percent"),
    col("parsed.main.pressure").alias("pressure_hpa"),

    col("parsed.wind.speed").alias("wind_speed_mps"),
    col("parsed.weather")[0]["description"].alias("weather_description"),

    col("parsed.rain.1h").alias("rainfall_mm"),

    from_unixtime(col("parsed.dt")).cast("timestamp").alias("event_timestamp"),
    from_unixtime(col("parsed.sys.sunrise")).cast("timestamp").alias("sunrise_time"),
    from_unixtime(col("parsed.sys.sunset")).cast("timestamp").alias("sunset_time"),

    col("call_timestamp").alias("ingestion_timestamp")
)


In [0]:
weather_df = weather_df.fillna({"rainfall_mm": 0.0})

# Derived columns
weather_df = weather_df \
    .withColumn("event_date", to_date("event_timestamp")) \
    .withColumn("event_hour", hour("event_timestamp")) \
    .withColumn("temperature_band",
        when(col("temperature_celsius") < 15, "Cold")
        .when((col("temperature_celsius") <= 30), "Normal")
        .when((col("temperature_celsius") <= 40), "Hot")
        .otherwise("Extreme")
    ) \
    .withColumn("weather_severity_score",
        (when(col("temperature_celsius") > 40, 3).otherwise(0)) +
        (when(col("rainfall_mm") > 30, 2).otherwise(0)) +
        (when(col("wind_speed_mps") > 40, 2).otherwise(0))
    ) \
    .withColumn("is_daytime",
        when(
            col("sunrise_time").isNull() | col("sunset_time").isNull(),
            None
        ).when(
            (col("event_timestamp") >= col("sunrise_time")) &
            (col("event_timestamp") <= col("sunset_time")),
            True
        ).otherwise(False)
    ) \
    .withColumn("source_system", lit("openweathermap")) \
    .withColumn("ingestion_time", current_timestamp())

In [0]:
def process_batch(batch_df, batch_id):

    print(f"Processing batch {batch_id}")
    print("Columns:", batch_df.columns)


    df_with_reason = batch_df.withColumn(
        "quarantine_reason",

        # API error
        when(col("http_status") != 200, "api_error")

        # Unparseable JSON
        .when(col("city_name").isNull(), "unparseable_json")

        # Extreme temperature
        .when((col("temperature_celsius") > 60) | (col("temperature_celsius") < -50),
              "out_of_range_temperature")

        # Humidity
        .when((col("humidity_percent") > 100) | (col("humidity_percent") < 0),
              "out_of_range_humidity")

        # Wind
        .when(col("wind_speed_mps") < 0, "out_of_range_wind_speed")

        # Null city
        .when(trim(col("city_name")) == "", "null_city")

        # Future timestamp
        .when(col("event_timestamp") > current_timestamp() + expr("INTERVAL 1 HOUR"),
              "future_event_time")

        # Missing coordinates
        .when(col("latitude").isNull() | col("longitude").isNull(),
              "missing_coordinates")


        .otherwise(None)
    )

    valid_df = df_with_reason.filter(col("quarantine_reason").isNull())
    quarantine_df = df_with_reason.filter(col("quarantine_reason").isNotNull())

    # Write clean
    valid_df.write.format("delta") \
        .mode("append") \
        .partitionBy("event_date") \
        .saveAsTable("realtime_weather.silver.silver_weather_clean")

    # Write quarantine
    quarantine_df.select(
    "city_name",
    "raw_response",
    col("ingestion_timestamp").alias("call_timestamp"),
    "quarantine_reason"
).withColumn("quarantined_at", current_timestamp()) \
 .write.format("delta") \
 .mode("append") \
 .saveAsTable("realtime_weather.silver.silver_weather_quarantine")


In [0]:
query = weather_df.writeStream \
    .foreachBatch(process_batch) \
    .option("checkpointLocation", "/Volumes/realtime_weather/silver/checkpoint") \
    .trigger(availableNow=True) \
    .start()

query.awaitTermination()

In [0]:
%sql
ALTER TABLE realtime_weather.silver.silver_weather_clean
SET TBLPROPERTIES (
  delta.enableChangeDataFeed = true
);

In [0]:
display(spark.table("realtime_weather.silver.silver_weather_clean"))
display(spark.table("realtime_weather.silver.silver_weather_quarantine"))

http_status,raw_response,call_timestamp,city_name,country,latitude,longitude,temperature_celsius,feels_like_celsius,humidity_percent,pressure_hpa,wind_speed_mps,weather_description,rainfall_mm,event_timestamp,sunrise_time,sunset_time,ingestion_timestamp,event_date,event_hour,temperature_band,weather_severity_score,is_daytime,source_system,ingestion_time,quarantine_reason
200,"{""coord"":{""lon"":80.2785,""lat"":13.0878},""weather"":[{""id"":801,""main"":""Clouds"",""description"":""few clouds"",""icon"":""02d""}],""base"":""stations"",""main"":{""temp"":35.1,""feels_like"":42.1,""temp_min"":34.98,""temp_max"":36.24,""pressure"":1005,""humidity"":60,""sea_level"":1005,""grnd_level"":1005},""visibility"":6000,""wind"":{""speed"":7.72,""deg"":170},""clouds"":{""all"":20},""dt"":1777454829,""sys"":{""type"":2,""id"":2104103,""country"":""IN"",""sunrise"":1777421939,""sunset"":1777467201},""timezone"":19800,""id"":1264527,""name"":""Chennai"",""cod"":200}",2026-04-29T09:30:34.757Z,Chennai,IN,13.0878,80.2785,35.1,42.1,60,1005,7.72,few clouds,0.0,2026-04-29T09:27:09.000Z,2026-04-29T00:18:59.000Z,2026-04-29T12:53:21.000Z,2026-04-29T09:30:34.757Z,2026-04-29,9,Hot,0,true,openweathermap,2026-04-30T05:49:35.735Z,null
200,"{""coord"":{""lon"":72.8479,""lat"":19.0144},""weather"":[{""id"":721,""main"":""Haze"",""description"":""haze"",""icon"":""50d""}],""base"":""stations"",""main"":{""temp"":32.99,""feels_like"":38.79,""temp_min"":31.94,""temp_max"":32.99,""pressure"":1006,""humidity"":58,""sea_level"":1006,""grnd_level"":1006},""visibility"":6000,""wind"":{""speed"":7.72,""deg"":230},""clouds"":{""all"":0},""dt"":1777454683,""sys"":{""type"":1,""id"":9052,""country"":""IN"",""sunrise"":1777423318,""sunset"":1777469388},""timezone"":19800,""id"":1275339,""name"":""Mumbai"",""cod"":200}",2026-04-29T09:30:34.831Z,Mumbai,IN,19.0144,72.8479,32.99,38.79,58,1006,7.72,haze,0.0,2026-04-29T09:24:43.000Z,2026-04-29T00:41:58.000Z,2026-04-29T13:29:48.000Z,2026-04-29T09:30:34.831Z,2026-04-29,9,Hot,0,true,openweathermap,2026-04-30T05:49:35.735Z,null
200,"{""coord"":{""lon"":77.2167,""lat"":28.6667},""weather"":[{""id"":802,""main"":""Clouds"",""description"":""scattered clouds"",""icon"":""03d""}],""base"":""stations"",""main"":{""temp"":36.05,""feels_like"":35.76,""temp_min"":36.05,""temp_max"":36.05,""pressure"":1000,""humidity"":28,""sea_level"":1000,""grnd_level"":976},""visibility"":6000,""wind"":{""speed"":3.09,""deg"":80},""clouds"":{""all"":40},""dt"":1777454982,""sys"":{""type"":1,""id"":9165,""country"":""IN"",""sunrise"":1777421535,""sunset"":1777469075},""timezone"":19800,""id"":1273294,""name"":""Delhi"",""cod"":200}",2026-04-29T09:30:34.902Z,Delhi,IN,28.6667,77.2167,36.05,35.76,28,1000,3.09,scattered clouds,0.0,2026-04-29T09:29:42.000Z,2026-04-29T00:12:15.000Z,2026-04-29T13:24:35.000Z,2026-04-29T09:30:34.902Z,2026-04-29,9,Hot,0,true,openweathermap,2026-04-30T05:49:35.735Z,null
200,"{""coord"":{""lon"":77.6033,""lat"":12.9762},""weather"":[{""id"":802,""main"":""Clouds"",""description"":""scattered clouds"",""icon"":""03d""}],""base"":""stations"",""main"":{""temp"":34.72,""feels_like"":36.7,""temp_min"":32.9,""temp_max"":35.76,""pressure"":1005,""humidity"":40,""sea_level"":1005,""grnd_level"":910},""visibility"":6000,""wind"":{""speed"":6.71,""deg"":108,""gust"":22.8},""clouds"":{""all"":40},""dt"":1777454643,""sys"":{""type"":2,""id"":2017753,""country"":""IN"",""sunrise"":1777422588,""sunset"":1777467836},""timezone"":19800,""id"":1277333,""name"":""Bengaluru"",""cod"":200}",2026-04-29T09:30:34.970Z,Bengaluru,IN,12.9762,77.6033,34.72,36.7,40,1005,6.71,scattered clouds,0.0,2026-04-29T09:24:03.000Z,2026-04-29T00:29:48.000Z,2026-04-29T13:03:56.000Z,2026-04-29T09:30:34.970Z,2026-04-29,9,Hot,0,true,openweathermap,2026-04-30T05:49:35.735Z,null
200,"{""coord"":{""lon"":78.4744,""lat"":17.3753},""weather"":[{""id"":801,""main"":""Clouds"",""description"":""few clouds"",""icon"":""02d""}],""base"":""stations"",""main"":{""temp"":38.23,""feels_l

city_name,raw_response,call_timestamp,quarantine_reason,quarantined_at
